# 01a — Auto-Caption Reference Images

Generates `.txt` sidecar caption files for each reference image using JoyCaption.
These captions are used for SDXL LoRA training in `01b_train_sdxl_lora.ipynb`.

**Runtime:** T4 is fine (captioning is light). A100 optional.

**Prerequisites:** JoyCaption is a gated model — you need a free HuggingFace account.
1. Sign up at https://huggingface.co (free)
2. Accept the model terms at https://huggingface.co/fancyfeast/llama-joycaption-beta-one-hf-llava
3. Create a read token at https://huggingface.co/settings/tokens
4. Paste it in Cell 1b below when prompted

**Alternative:** If you don't want to create an HF account, use the `llava-hf/llava-1.5-7b-hf` 
model instead (fully public, similar caption quality). Change `USE_JOYCAPTION = True` to `False` in Cell 1b.

**Flow:**
1. HuggingFace login
2. Upload reference images to Drive (or Colab tmp)
3. Run captioner on each image
4. Prepend trigger token to each caption
5. Save `.txt` sidecar files next to each image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
CHARACTER_NAME = 'Aria'       # ← change this
TRIGGER_TOKEN  = 'ohwx_aria'  # ← change this (must match what you use in training)

# Captioner choice:
# False = LLaVA 1.5 7B (default — fully public, works out-of-the-box, great for LoRA training)
# True  = JoyCaption Beta One (highest quality, but requires HF account + model access at
#         https://huggingface.co/fancyfeast/llama-joycaption-beta-one-hf-llava)
USE_JOYCAPTION = False

import os
CHAR_DIR    = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}'
REF_DIR     = f'{CHAR_DIR}/reference-images'
CAPTION_DIR = f'{CHAR_DIR}/captions'
os.makedirs(REF_DIR, exist_ok=True)
os.makedirs(CAPTION_DIR, exist_ok=True)

print(f'Character: {CHARACTER_NAME} / trigger: {TRIGGER_TOKEN}')
print(f'Reference images dir: {REF_DIR}')
print(f'Captioner: {"JoyCaption Beta One" if USE_JOYCAPTION else "LLaVA 1.5 7B (public, default)"}')
print('Upload your reference images to the Drive folder above, then run the next cells.')

In [ ]:
# (Optional) Upload images directly from this notebook
from google.colab import files
import shutil

print('Select your reference images to upload...')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f'{REF_DIR}/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved {fname} → {dest}')

In [ ]:
!pip install -q transformers torch Pillow huggingface_hub

from transformers import AutoProcessor, LlavaForConditionalGeneration
import torch, os

if USE_JOYCAPTION:
    # JoyCaption Beta One — gated model. Two ways to authenticate:
    # Option 1 (recommended): Add HF_TOKEN to Colab Secrets (key icon in left sidebar → "Add secret")
    # Option 2: paste token directly below (less secure — don't commit this)
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
        print('HF_TOKEN loaded from Colab Secrets.')
    except Exception:
        # Fallback: paste token here
        from getpass import getpass
        hf_token = getpass('Paste your HuggingFace read token (hidden): ')

    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    MODEL_ID = 'fancyfeast/llama-joycaption-beta-one-hf-llava'
else:
    MODEL_ID = 'llava-hf/llava-1.5-7b-hf'

print(f'Loading captioner: {MODEL_ID} ...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16
).to('cuda' if torch.cuda.is_available() else 'cpu')
print('Captioner loaded.')

In [ ]:
from PIL import Image
from pathlib import Path

# ── DIAGNOSTIC — run this first to see what token/format works ──────────────
print('=== JoyCaption diagnostics ===')
print('image_token attr:', repr(getattr(processor, 'image_token', 'NOT FOUND')))
print('tokenizer image_token:', repr(getattr(processor.tokenizer, 'image_token', 'NOT FOUND')))
extra = getattr(processor.tokenizer, 'additional_special_tokens', [])
print('additional_special_tokens:', extra[:6])

# Find a test image
test_imgs = [p for p in Path(REF_DIR).iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
if test_imgs:
    test_img = test_imgs[0]
    test_pil = Image.open(test_img).convert('RGB')
    print(f'\nTest image: {test_img.name}')

    # Try every prompt format and print the result
    img_tok = getattr(processor, 'image_token', None) or getattr(processor.tokenizer, 'image_token', '<image>') or '<image>'
    print(f'Using image_token: {repr(img_tok)}')

    for label, conv in [
        ('list_content',   [{"role":"user","content":[{"type":"image"},{"type":"text","text":"test"}]}]),
        ('string_content', [{"role":"user","content":f"{img_tok}\ntest"}]),
    ]:
        try:
            p = processor.apply_chat_template(conv, add_generation_prompt=True)
            has_tok = img_tok in p
            inputs = processor(text=p, images=[test_pil], return_tensors='pt')
            n_img_toks = (inputs['input_ids'] == processor.tokenizer.convert_tokens_to_ids(img_tok)).sum().item() \
                         if img_tok in processor.tokenizer.get_vocab() else -1
            print(f'  {label}: OK | img_tok_in_prompt={has_tok} | img_tok_in_ids={n_img_toks} | prompt[:80]={repr(p[:80])}')
        except Exception as e:
            print(f'  {label}: FAILED — {e}')

print('\n=== End diagnostics — read the output above, then proceed ===')


In [ ]:
# Review and optionally edit captions before training
print('=== Caption Review ===')
for fname, caption in captions.items():
    print(f'\n[{fname}]')
    print(caption)
print('\nEdit the .txt files in Drive if any captions need adjustment before training.')

In [ ]:
# Register character in library (optional — also done in training notebook)
import sys, json
# If running from Colab, library.py isn't installed — use inline version
metadata = {
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'sdxl',
    'ref_count': len(captions),
}
meta_path = f'{CHAR_DIR}/metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'metadata.json written: {meta_path}')
print('\n✅ Done. Run 01b_train_sdxl_lora.ipynb next to train the character LoRA.')